# Результаты бенчмарков rangequery

На графиках ось X — точный размер множества `data_size`. Размеры увеличены до 100 000 ключей; для каждого размера все четыре структуры получают одинаковый входной поток.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BINARY = ROOT / 'build' / 'release' / 'src' / 'rangequery'
STRUCTURES = ('vector', 'set', 'avl', 'treap')
RESULTS = ROOT / 'benchmark-results' / 'comparison.csv'
RESULTS.parent.mkdir(exist_ok=True)
subprocess.run([sys.executable, str(ROOT / 'tools' / 'benchmark.py'), '--binary', str(BINARY), '--output', str(RESULTS), '--scenario', 'comparison', '--data-sizes', '1000', '10000', '100000', '--query-count', '1000', '--query-kind', 'rank', '--density', 'sparse', '--repetitions', '3', '--seed', '42'], check=True)
results = pd.read_csv(RESULTS)
results.head()


In [ ]:
summary = results.groupby(['structure', 'data_size'], as_index=False)['ns_per_operation'].median()
for structure, group in summary.groupby('structure'):
    plt.loglog(group['data_size'], group['ns_per_operation'], marker='o', label=structure)
plt.xlabel('размер множества'); plt.ylabel('медианное время команды (нс)'); plt.title('Разреженные данные: одинаковые rank-запросы')
plt.grid(True, which='both', alpha=.25); plt.legend(); plt.show()

## Режимы данных

Эти графики не подбирают отдельную операцию для каждой структуры. Внутри каждого режима все реализации получают один и тот же входной поток.

| Режим | Размеры | Вопрос |
| --- | --- | --- |
| `small_dense_rank` | 1 000–10 000 | Как ведут себя структуры на небольших плотных множествах? |
| `medium_dense_rank` / `medium_sparse_rank` | 10 000–100 000 | Влияет ли плотность ключей при одинаковой нагрузке? |
| `medium_sparse_mixed` | 10 000–100 000 | Что происходит в сбалансированном чтении: `rank`, `select`, интервал? |
| `large_sparse_*_growth` | 10 000–100 000 | Как меняется цена роста при случайном, возрастающем и убывающем поступлении ключей? |


In [ ]:
regimes_path = ROOT / 'benchmark-results' / 'regimes.csv'
subprocess.run([sys.executable, str(ROOT / 'tools' / 'benchmark.py'), '--binary', str(BINARY), '--output', str(regimes_path), '--scenario', 'regimes', '--repetitions', '3', '--seed', '42'], check=True)
regimes = pd.read_csv(regimes_path)
regimes_summary = regimes.groupby(['scenario', 'structure', 'data_size', 'density', 'operation_kind', 'insertion_order'], as_index=False)['ns_per_operation'].median()
scenario_names = list(regimes_summary['scenario'].unique())
columns = 3
rows = (len(scenario_names) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(6 * columns, 4 * rows), squeeze=False)
for axis, scenario in zip(axes.flat, scenario_names):
    group = regimes_summary[regimes_summary['scenario'] == scenario]
    for structure, series in group.groupby('structure'):
        axis.loglog(series['data_size'], series['ns_per_operation'], marker='o', label=structure)
    row = group.iloc[0]
    details = row['insertion_order'] if row['operation_kind'] == 'insert' else row['operation_kind']
    axis.set_title(f"{scenario}: {row['density']}, {details}")
    axis.set_xlabel('размер множества')
    axis.set_ylabel('медианное время команды (нс)')
    axis.grid(True, which='both', alpha=.25)
for axis in axes.flat[len(scenario_names):]:
    axis.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(labels))
fig.suptitle('Режимы размера, плотности и порядка вставки', y=1.02)
fig.tight_layout()


### Контроль плотности

`medium_dense_rank` и `medium_sparse_rank` имеют одинаковые размеры и одинаковые rank-запросы. Разница между кривыми показывает эффект распределения самих ключей, а не смену операции или масштаба.


In [ ]:
density_profiles = ('medium_dense_rank', 'medium_sparse_rank')
density_data = regimes_summary[regimes_summary['scenario'].isin(density_profiles)]
fig, axes = plt.subplots(1, len(STRUCTURES), figsize=(5 * len(STRUCTURES), 4), sharey=True)
for axis, structure in zip(axes, STRUCTURES):
    group = density_data[density_data['structure'] == structure]
    for density, series in group.groupby('density'):
        axis.loglog(series['data_size'], series['ns_per_operation'], marker='o', label=density)
    axis.set_title(structure)
    axis.set_xlabel('размер множества')
    axis.grid(True, which='both', alpha=.25)
axes[0].set_ylabel('медианное время команды (нс)')
axes[-1].legend(title='плотность')
fig.suptitle('Контроль влияния плотности ключей', y=1.03)
fig.tight_layout()


### Порядок роста

Здесь измеряется один и тот же набор разреженных ключей, но меняется порядок их поступления. Этот тест отделяет стоимость поиска позиции и сдвигов от эффекта числовой плотности.


In [ ]:
growth_profiles = ('large_sparse_random_growth', 'large_sparse_ascending_growth', 'large_sparse_descending_growth')
growth_labels = {'random': 'случайный', 'ascending': 'возрастающий', 'descending': 'убывающий'}
growth_data = regimes_summary[regimes_summary['scenario'].isin(growth_profiles)]
fig, axes = plt.subplots(1, len(STRUCTURES), figsize=(5 * len(STRUCTURES), 4), sharey=True)
for axis, structure in zip(axes, STRUCTURES):
    group = growth_data[growth_data['structure'] == structure]
    for order, series in group.groupby('insertion_order'):
        axis.loglog(series['data_size'], series['ns_per_operation'], marker='o', label=growth_labels[order])
    axis.set_title(structure)
    axis.set_xlabel('размер множества')
    axis.grid(True, which='both', alpha=.25)
axes[0].set_ylabel('медианное время команды (нс)')
axes[-1].legend(title='порядок вставки')
fig.suptitle('Рост разреженного множества: влияние порядка вставки', y=1.03)
fig.tight_layout()


## Локальность обращений

Здесь выполняются 100 000 rank-запросов на множествах среднего размера. `uniform` обращается ко всему диапазону ключей, а `hotspot` концентрирует обращения вокруг центральных 5 % ключей.

In [ ]:
LOCALITY_SIZES = ('1000', '3000', '10000')
LOCALITY_QUERY_COUNT = '100000'
locality_results_path = ROOT / 'benchmark-results' / 'locality.csv'
subprocess.run([sys.executable, str(ROOT / 'tools' / 'benchmark.py'), '--binary', str(BINARY), '--output', str(locality_results_path), '--scenario', 'comparison', '--data-sizes', *LOCALITY_SIZES, '--query-count', LOCALITY_QUERY_COUNT, '--query-kind', 'rank', '--localities', 'uniform', 'hotspot', '--density', 'sparse', '--repetitions', '3', '--seed', '42'], check=True)
locality_results = pd.read_csv(locality_results_path)
locality_summary = locality_results.groupby(['structure', 'locality', 'data_size'], as_index=False)['ns_per_operation'].median()
structures = list(locality_summary['structure'].unique())
fig, axes = plt.subplots(1, len(structures), figsize=(5 * len(structures), 4), sharey=True)
for axis, structure in zip(axes, structures):
    group = locality_summary[locality_summary['structure'] == structure]
    for locality, series in group.groupby('locality'):
        axis.loglog(series['data_size'], series['ns_per_operation'], marker='o', label=locality)
    axis.set_title(structure); axis.set_xlabel('размер множества'); axis.grid(True, which='both', alpha=.25)
axes[0].set_ylabel('медианное время команды (нс)'); axes[-1].legend(title='локальность')
fig.suptitle('Влияние локальности обращений: 100 000 rank-запросов'); fig.tight_layout(); plt.show()